In [1]:
import torch
import cv2
import numpy as np

import os
from extraction.images import model_loader
from extraction.video_processing import extract_video_features_compressed_ms

import time

In [2]:
harmful_dir = ['data/images/tikharm/train/Harmful Content']
safe_dir = ['data/images/tikharm/train/Safe']

In [3]:
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

In [4]:
# 1. Device Guard Setup
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"⏳ Loading transformer weights into active hardware storage space...")

# 2. Initialize weights ONCE at the top level of the cell
global_model, global_processor, active_code = model_loader(model_code='clip', device=device)
print(f"✅ Transformer successfully cached on: {device.upper()}\n")

⏳ Loading transformer weights into active hardware storage space...
✅ Transformer successfully cached on: MPS



In [5]:
harmful_data = []

# ─── PROCESS VIOLENT DATASET TRACK ───
for video_dir in harmful_dir[:600]:
    if not os.path.isdir(video_dir): continue
        
    for video_file in os.listdir(video_dir):
        # Skip system garbage metadata files like .DS_Store
        if video_file.startswith('.'): continue 

        video_path = os.path.join(video_dir, video_file)
        start_timer = time.time()
        
        # 🔥 FIXED: Passing global_model and global_processor variables smoothly instead of string codes
        vector = extract_video_features_compressed_ms(
            video_path=video_path,
            model=global_model,
            processor=global_processor,
            model_code=active_code,
            target_frames=18
        )
        
        harmful_data.append(vector)
        execution_speed = time.time() - start_timer

[NULL @ 0x32fffbdf0] Invalid NAL unit size (16765 > 3318).
[NULL @ 0x32fffbdf0] missing picture in access unit with size 3322
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x336c05d60] stream 1, offset 0x3683087: partial file
[h264 @ 0x32fff7f80] Invalid NAL unit size (16765 > 3318).
[h264 @ 0x32fff7f80] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x336c05d60] stream 0, offset 0x368315e: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x336c05d60] stream 0, offset 0x3b38ec5: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x336c05d60] stream 0, offset 0x3b38ec5: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x336c05d60] stream 0, offset 0x3b38ec5: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x336c05d60] stream 0, offset 0x3b38ec5: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x336c05d60] stream 1, offset 0x393e3e6: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x336c05d60] stream 1, offset 0x393e3e6: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x336c05d60] stream 1, offset 0x393e3e6: partial file
[mov,mp4,m4a,3gp,3g2,mj

In [6]:
safe_data = []

# ─── PROCESS VIOLENT DATASET TRACK ───
for video_dir in safe_dir:
    if not os.path.isdir(video_dir): continue
    
    for video_file in os.listdir(video_dir)[:150]:
        # Skip system garbage metadata files like .DS_Store
        if video_file.startswith('.'): continue 

        video_path = os.path.join(video_dir, video_file)      
        start_timer = time.time()
        
        # 🔥 FIXED: Passing global_model and global_processor variables smoothly instead of string codes
        vector = extract_video_features_compressed_ms(
            video_path=video_path,
            model=global_model,
            processor=global_processor,
            model_code=active_code,
            target_frames=18
        )
        
        safe_data.append(vector)
        execution_speed = time.time() - start_timer

In [7]:
from sklearn.model_selection import train_test_split
from training.kfold_train import stratified_kfold_train_val

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.metrics import confusion_matrix, classification_report

In [8]:
harmful_labels = np.ones(len(harmful_data))
safe_labels = np.zeros(len(safe_data))

harmful_data = np.array(harmful_data)
safe_data = np.array(safe_data)

In [9]:
x_harmful = np.concat([harmful_data, safe_data])
y_harmful = np.concat([harmful_labels, safe_labels])

In [10]:
x_train, x_test, y_train, y_test = train_test_split(x_harmful, y_harmful,
                                                    stratify=y_harmful,
                                                    test_size=0.2)

In [11]:
log_reg = LogisticRegression(
    penalty='l2',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='liblinear',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)

stratified_kfold_train_val(5,
                           0.26,
                           log_reg,
                           x_train,
                           y_train)

Starting 5-Fold Stratified Cross-Validation...

sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9550 (When flagged positive, accuracy is 95.50%)
Custom Recall Score:    0.9550 (Captured 95.50% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9730 (When flagged positive, accuracy is 97.30%)
Custom Recall Score:    0.9730 (Captured 97.30% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9727 (When flagged positive, accuracy is 97.27%)
Custom Recall Score:    0.9727 (Captured 97.27% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9649 (When flagged positive, accuracy is 96.49%)
Custom Recall Score:    1.0000 (Captured 100.00% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9725 (When flagged positive, accuracy is 97.25%)
Custom 

In [12]:
log_reg = LogisticRegression(
    penalty='l2',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='liblinear',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)
threshold = 0.26

log_reg.fit(x_train, y_train)
test_probabilities = log_reg.predict_proba(x_test)[:, 1]
predictions = (test_probabilities >= threshold).astype(int)

print(classification_report(y_test, predictions))
print(confusion_matrix(y_test, predictions))

              precision    recall  f1-score   support

         0.0       0.80      0.80      0.80        30
         1.0       0.96      0.96      0.96       138

    accuracy                           0.93       168
   macro avg       0.88      0.88      0.88       168
weighted avg       0.93      0.93      0.93       168

[[ 24   6]
 [  6 132]]


In [13]:
import joblib

joblib.dump(log_reg, open("modelling/model/harmful.jobllib", 'wb'))

In [15]:
np.savez("data/images/harmful.npz", features=x_harmful, labels=y_harmful)

In [44]:
from xgboost import XGBClassifier